In [1]:
!pip install nltk pandas

In [2]:
import json
import re
from collections import Counter
from pathlib import Path
import nltk
from nltk.corpus import words as nltk_words

# Download English word dictionary if not present
nltk.download("words")

DATA_DIR = Path("/Users/viv/Downloads/Ayurveda")
INPUT_FILE = DATA_DIR / "result.json"

# Fast lookup set of lowercase English words
english_vocab = set(w.lower() for w in nltk_words.words())

# Add common English grammatical variants/inflections often missing from basic wordnet/words
english_vocab.update({
    "ing", "ed", "ly", "er", "est", "s", "es", "al", "ic", "tion", "sion"
})

print(f"Loaded {len(english_vocab)} standard English reference words.")

[nltk_data] Downloading package words to /Users/viv/nltk_data...


Loaded 234380 standard English reference words.


[nltk_data]   Unzipping corpora/words.zip.


In [3]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

# Extract all word tokens containing only alphabets (length >= 3)
TOKEN_PATTERN = re.compile(r"\b[A-Za-z]{3,}\b")

non_english_counter = Counter()

for chunk in chunks:
    content = chunk.get("content", "")
    tokens = TOKEN_PATTERN.findall(content)
    
    for token in tokens:
        word = token.lower()
        # If token is not in English dictionary, flag it as a domain candidate
        if word not in english_vocab:
            non_english_counter[word] += 1

print(f"Discovered {len(non_english_counter)} candidate non-English/Sanskrit terms.")

Discovered 47001 candidate non-English/Sanskrit terms.


In [4]:
import pandas as pd

LEXICON_CSV = DATA_DIR / "ayurveda_lexicon.csv"
LEXICON_TXT = DATA_DIR / "ayurveda_lexicon.txt"

# Convert to DataFrame sorted by occurrence frequency
df = pd.DataFrame(non_english_counter.most_common(), columns=["term", "frequency"])

# Filter out low-frequency OCR artifacts (e.g. min frequency >= 2)
df_filtered = df[df["frequency"] >= 2]

# 1. Export CSV (term, frequency)
df_filtered.to_csv(LEXICON_CSV, index=False)

# 2. Export TXT (one term per line)
with open(LEXICON_TXT, "w", encoding="utf-8") as f:
    for term in df_filtered["term"]:
        f.write(f"{term}\n")

print(f"Saved {len(df_filtered)} terms to:")
print(f" - CSV: {LEXICON_CSV}")
print(f" - TXT: {LEXICON_TXT}")

# Preview the top 20 extracted domain terms
df_filtered.head(20)

Saved 16764 terms to:
 - CSV: /Users/viv/Downloads/Ayurveda/ayurveda_lexicon.csv
 - TXT: /Users/viv/Downloads/Ayurveda/ayurveda_lexicon.txt


,term,frequency
0,has,2316
1,etc,1438
2,drugs,1402
3,ayurveda,1356
4,ayurvedic,1142
5,called,1104
6,kapha,1077
7,diseases,1056
8,cases,968
9,having,966
